# The history matching workflow

History matching adjusts uncertain model parameters so that the model's predictions agree with observed data. In APEX you describe the problem in a TOML configuration file, validate it, run it, and read the posterior summary.

**Two methods, and why we compare them**

- **ES-MDA** (Ensemble Smoother with Multiple Data Assimilation) keeps an *ensemble*, a set of candidate models. It updates the whole ensemble a fixed number of times against the same data. The data uncertainty is inflated at each pass, so every pass takes a gentler step.
- **GIES** is an iterative ensemble method. Its update is a regularised Levenberg-Marquardt step, selected in the configuration by the strategy name `rlmmac`. You do not need its theory here. GIES adapts its steps through a damping scheme and stops when its own criteria are met or an iteration cap is reached.

Both methods solve the same problem here, so we can run them side by side.

**What you will do**

1. Copy a ready-made example from the examples catalogue.
2. Read its ES-MDA configuration.
3. Validate it and read the plan.
4. Run it, wait for it, and read the posterior metrics.
5. Repeat with GIES on the same problem and compare the two.
6. Find the run directories and inspect them from the terminal, then stop and resume a short run.

The example, *Linear-Gaussian: an exact posterior*, has a linear model with a Gaussian prior and Gaussian data. It is self-contained, so no simulator knowledge is needed. It is also *synthetic*: the data were generated from a known true model, which lets APEX score the result against that truth.

**Prerequisites**

- APEX is installed. Run this notebook from the APEX folder (the folder where you type `uv run apex ...`). It creates the folders `work` and `apex_home` there, and the terminal commands further down use paths relative to the same folder. A folder of its own keeps this notebook's runs apart from other work.
- You know roughly what a prior, an observation and a posterior are.

## Step 1: copy an example to work on

We look the example up by its id, `linear_gaussian`, and copy it into a `work` folder. You can then edit your copy freely.

The example holds two configurations for the same problem: `config_esmda.toml` (method ES-MDA) and `config_gies.toml` (method GIES).

In [1]:
from pathlib import Path

from apex.client import Client
from apex.services.examples import copy_example, examples_root, find_examples
from apex.services.settings import ExamplesSection

# The examples catalogue: copy one to work on.
catalogue = {e.id: e for e in find_examples(examples_root(ExamplesSection()))}
copied = copy_example(catalogue["linear_gaussian"], Path("work"))
config = copied.folder / "config_esmda.toml"
print(config.relative_to(Path.cwd()))

work/linear_gaussian/config_esmda.toml


## Step 2: read the configuration

A run is fully described by its configuration file, so read it before you run it. The next cell prints the file without comments and blank lines. The sections mean:

- `[run]`: `name` labels the run, and `seed` fixes the random numbers so that the run can be repeated.
- `[[state.variables]]`: the unknowns we estimate. Here one variable `x` with three values.
- `[prior]`: the starting ensemble. `ensemble_size` is the number of members, and `generator` draws them, here independent normals (`gaussian`) with a `mean` and a `std`.
- `[observations]` and `[case]`: the data to match, and the true model (`truth_file`) used for scoring because this case is `synthetic`.
- `[simulator]`: the forward model. The `linear` simulator computes `y = G x` for a fixed matrix `G` (here read from `forward_file`), mapping the three values of `x` to eight outputs.
- `[execution]`: how simulations are run. `workers` sets how many run at the same time. It changes speed, not the problem.
- `[method]`: the algorithm and its options. The `inflation` list of ES-MDA has one entry per step, and each step is one *analysis step*: one update of the whole ensemble against the data. `strategy` names the update rule used inside each step. Here it is `approx`, with an `energy` option: the fraction of the singular-value sum of the data anomalies that the update retains, a number in (0, 1]. `energy = 1.0` keeps everything, with no truncation. You can leave it at that value for this tutorial.

When you want to experiment, edit a copy of the configuration, give it another `[run]` name, and validate it before running it.

In [2]:
def setting(line):
    for index, char in enumerate(line):
        if char == "#" and line[:index].count('"') % 2 == 0:
            return line[:index].rstrip()
    return line.rstrip()


settings = [setting(line) for line in config.read_text().splitlines()]
print("\n".join(line for line in settings if line))

[run]
name = "linear_esmda"
seed = 2026
[[state.variables]]
name = "x"
size = 3
[prior]
ensemble_size = 500
generator = { name = "gaussian", options = { mean = 0.0, std = 1.0 } }
[observations]
file = "observations.csv"
[case]
kind = "synthetic"
truth_file = "truth.npz"
[simulator]
name = "linear"
[simulator.options]
forward_file = "forward.npz"
variables = ["x"]
outputs = [["t0", "y"], ["t1", "y"], ["t2", "y"], ["t3", "y"], ["t4", "y"], ["t5", "y"], ["t6", "y"], ["t7", "y"]]
[execution]
name = "process"
options = { workers = 4 }
[method]
name = "esmda"
options = { inflation = [4, 4, 4, 4] }
strategy = { name = "approx", options = { energy = 1.0 } }


## Step 3: validate and look at the plan

`Client(Path("apex_home"))` is a client on a services root of this notebook's own: the folder `apex_home`. It keeps the records and logs of what you submit through this client, so they stay apart from any other work. (The run directories themselves are elsewhere; Step 7 shows where.)

`validate` checks the configuration and plans its cost, and nothing is simulated. The plan reports the method, the number of members and the most simulator runs the configuration could use. For an expensive simulator that maximum is the most you could pay, so read it before you start a run.

For ES-MDA the number of analysis steps is the length of the `inflation` list, and iteration 0 is the prior ensemble. The number of ensembles simulated is that length plus one, so the maximum is (4 + 1) × 500 = 2500.

In [3]:
client = Client(Path("apex_home"))  # a services root of its own
checked = client.validate(config)
print("valid:", checked.valid)
plan = checked.plan
print(f"{plan.method}: {plan.members} members, at most {plan.max_simulator_runs} simulator runs")

valid: True
esmda: 500 members, at most 2500 simulator runs


## Step 4: run and wait

`submit_run` submits the configuration and gives back a handle. `wait` takes the handle's id and waits for the run to end, here for at most `timeout_s` seconds, polling every `poll_s` seconds. It returns a status whose `state` we print.

Read the state before you trust any summary: only `complete` is a finished run. If you see anything else, validate and read your configuration again, or raise `timeout_s` if the wait ran out.

In [4]:
handle = client.submit_run(config)
status = client.wait(handle.id, timeout_s=600, poll_s=0.5)
print(status.state)

complete


## Step 5: read the posterior summary

A submission has its own id, `handle.id`, and holds one or more runs, each with its own run id in `handle.run_ids`. Waiting uses the submission id. The summary belongs to a run, so it uses a run id. This example holds a single run, so we take index 0.

The iterations start at 0, the prior ensemble before any update. The last iteration is the posterior. The metrics are reported per variable and as `all`. The ones we read:

- `normalized_data_mismatch`: the mean of `((d_obs - y) / sigma)^2` over the data, that is, the squared distance between observed and simulated data in units of the observation uncertainty `sigma`. Lower is a closer fit.
- `data_mismatch`: a squared distance between observed and simulated data reported *without* the division by `sigma`. Its value is on a different scale from the normalized one, so do not compare the two numbers directly. Because every residual counts at its raw size, data with a large `sigma` weigh more here than they do in the normalized version, where each residual is measured against its own uncertainty. The two can therefore rank two runs differently. Use `normalized_data_mismatch` as the main measure of fit.
- `rmse`: the root-mean-square error of the parameters against the true model. It needs a known truth, which this synthetic case has. Lower is better.
- `model_change`: the change of each member's parameters since the ensemble it was updated from, so a rough size of the last step. It is not the distance from the prior.

In [5]:
summary = client.run_summary(handle.run_ids[0])
print(summary.method, summary.state, "iterations:", summary.iterations)
for name, values in summary.posterior_metrics.items():
    print(name, values)

esmda complete iterations: [0, 1, 2, 3, 4]
data_mismatch {'all': 18.92859148916899, 'y': 18.92859148916899}
normalized_data_mismatch {'all': 1.386665921591923, 'y': 1.386665921591923}
rmse {'x': 0.20097220595156404, 'all': 0.20097220595156404}
model_change {'x': 0.1040374288027935, 'all': 0.1040374288027935}


## Step 6: the same problem with GIES

The example also holds `config_gies.toml`: the same problem with method GIES. The next cell prints its `[method]` section and its plan. The section differs from ES-MDA:

- `strategy` named `rlmmac` selects GIES's update rule.
- `schedule` sets the options of the damping scheme (for GIES, the `rlmmac` scheme is the default). Damping controls the size of the update steps: more damping means smaller, more cautious steps. Options used here:
  - `initial` is the damping of the first analysis; it must be positive.
  - `factor` is what the damping is multiplied by after a rejected trial (greater than 1, default 2).
  - `data_mismatch_tolerance` is the relative change of the mean data mismatch below which an improving trial counts as converged (default 0.01). A smaller value lets the run continue until the improvements are tiny.
  - The scheme has further options with defaults of their own, for instance `reduction` (damping is multiplied by it after an accepted trial, default 0.9). Always read the values in your file, not the defaults.
- `max_iterations` caps the number of iterations.

GIES has no fixed iteration list: it stops when its own criteria are met or the cap is reached. The plan is therefore a ceiling for the worst case, and a run may stop much earlier. Take the ceiling from the plan that `validate` prints: do not rebuild it as `max_iterations` times the members, because the planner also allows for extra trials with more damping within an iteration.

In [6]:
gies_config = copied.folder / "config_gies.toml"
method_lines = []
in_method = False
for line in gies_config.read_text().splitlines():
    text = setting(line)
    if text.strip().startswith("["):
        in_method = text.strip() == "[method]"
    if in_method and text.strip():
        method_lines.append(text)
print("\n".join(method_lines))

gies_checked = client.validate(gies_config)
print("valid:", gies_checked.valid)
gies_plan = gies_checked.plan
print(
    f"{gies_plan.method}: {gies_plan.members} members, "
    f"at most {gies_plan.max_simulator_runs} simulator runs"
)

[method]
name = "gies"
strategy = { name = "rlmmac", options = { energy = 1.0 } }
schedule = { initial = 1.0, factor = 2.0, data_mismatch_tolerance = 1e-4 }
max_iterations = 20
valid: True
gies: 500 members, at most 10541 simulator runs


Now run it, and work out how much of the planned ceiling it used. Each ensemble simulated has one run per member, so the number of ensembles times the ensemble size gives the simulator runs used (when every iteration simulates one ensemble). Dividing by the ceiling shows the fraction spent.

In [7]:
gies_handle = client.submit_run(gies_config)
gies_status = client.wait(gies_handle.id, timeout_s=600, poll_s=0.5)
print(gies_status.state)

gies_summary = client.run_summary(gies_handle.run_ids[0])
print(gies_summary.method, gies_summary.state, "iterations:", gies_summary.iterations)
for name, values in gies_summary.posterior_metrics.items():
    print(name, values)

ensembles = len(gies_summary.iterations)
used = ensembles * gies_plan.members
print(f"ensembles simulated (including iteration 0): {ensembles}")
print(
    f"{ensembles} x {gies_plan.members} = {used} simulator runs, "
    f"against a planned maximum of {gies_plan.max_simulator_runs}"
)
print(f"fraction of the ceiling used: {used / gies_plan.max_simulator_runs:.0%}")

complete
gies complete iterations: [0, 1, 2, 3, 4, 5, 6, 7, 8]
data_mismatch {'all': 13.129146714443902, 'y': 13.129146714443902}
normalized_data_mismatch {'all': 1.4292791906552078, 'y': 1.4292791906552078}
rmse {'x': 0.20702499399688715, 'all': 0.20702499399688715}
model_change {'x': 0.002343765845202029, 'all': 0.002343765845202029}
ensembles simulated (including iteration 0): 9
9 x 500 = 4500 simulator runs, against a planned maximum of 10541
fraction of the ceiling used: 43%


The ceiling is a worst case; a run that converges early pays only for the ensembles it simulated.

## Comparing the two runs

Both runs start from the same prior and data, with the same ensemble size and seed, so metrics with the same name can be put side by side. This shows how the two posteriors compare on that problem. It does not make the runs equal in cost: they use different numbers of iterations and simulator runs. The table shows the `all` entry of each metric.

In [8]:
import pandas as pd

rows = {}
for name, values in summary.posterior_metrics.items():
    other = gies_summary.posterior_metrics.get(name, {})
    rows[name] = {"ES-MDA": values.get("all"), "GIES": other.get("all")}
table = pd.DataFrame(rows).T
print(table.to_string())

                             ES-MDA       GIES
data_mismatch             18.928591  13.129147
normalized_data_mismatch   1.386666   1.429279
rmse                       0.200972   0.207025
model_change               0.104037   0.002344


### Reading the table

The next cell reads the table metric by metric, saying which run is lower and by how much. Lower is better for the mismatch and `rmse` rows. `model_change` is only the size of the last step, so it is not ranked.

In [9]:
for name in table.index:
    if name == "model_change":
        continue
    a = table.loc[name, "ES-MDA"]
    g = table.loc[name, "GIES"]
    lower = "GIES" if g < a else "ES-MDA"
    gap = abs(g - a) / max(a, g) * 100
    print(f"{name}: lower for {lower}, by {gap:.1f}% of the larger value")

print(
    "model_change: ES-MDA",
    table.loc["model_change", "ES-MDA"],
    "GIES",
    table.loc["model_change", "GIES"],
)

data_mismatch: lower for GIES, by 30.6% of the larger value
normalized_data_mismatch: lower for ES-MDA, by 3.0% of the larger value
rmse: lower for ES-MDA, by 2.9% of the larger value
model_change: ES-MDA 0.1040374288027935 GIES 0.002343765845202029


How to read this output:

- **Mismatch rows**: the lower value fits the data more closely. The raw `data_mismatch` and the `normalized_data_mismatch` weigh the data differently (the second divides each residual by its observation uncertainty), so they can disagree on the ranking, as they may do here. Trust the normalized one, and when its gap is small, the runs differ little in fit and fit alone cannot pick a winner.
- **rmse**: how near each posterior is to the true model. We can read this only because the case is synthetic.
- **model_change**: a much smaller last step means a run had nearly stopped moving at its end. That says nothing about whether it is better or worse.

In a real case there is no truth, so compare methods through the fit, the iteration-by-iteration mismatch history from `apex report` (Step 8), the size of `model_change`, and above all the quality of the forecasts the posteriors give.

## Step 7: find the run directories

Each run has a directory of its own: its configuration snapshot, checkpoints, history and summary. The run summaries name it in `path`. For the runs of this notebook the directories are `work/linear_gaussian/runs/<name>`, where `<name>` is the `[run]` name: the run directory sits in a `runs` folder next to the configuration file.

In [10]:
def relative(path_text):
    path = Path(path_text)
    try:
        return path.relative_to(Path.cwd())
    except ValueError:
        return path


esmda_dir = relative(summary.path)
gies_dir = relative(gies_summary.path)
for name, run_dir in [("ES-MDA", esmda_dir), ("GIES", gies_dir)]:
    print(f"{name}: {run_dir}")
    print("   ", sorted(child.name for child in run_dir.iterdir()))

ES-MDA: work/linear_gaussian/runs/linear_esmda
    ['checkpoints', 'config.json', 'events.jsonl', 'history.json', 'provenance.json', 'run.lock', 'snapshot_version', 'state.json', 'summary.json']
GIES: work/linear_gaussian/runs/linear_gies
    ['checkpoints', 'config.json', 'events.jsonl', 'history.json', 'provenance.json', 'run.lock', 'snapshot_version', 'state.json', 'summary.json']


## Step 8: inspect from the terminal

Run these commands after the cells above have run, from the folder you ran the notebook in (the APEX folder). Their paths start at `work`, the folder the cells created there.

`apex validate` takes a configuration and optionally a services root with `--root`. It prints the plan again without simulating anything, so you should see the same method, member count and maximum number of simulator runs as in Step 6:

```bash
uv run apex validate work/linear_gaussian/config_gies.toml --root apex_home
```

`apex summary` takes run directories and lists every metric at every step. Here we pass the two directories from Step 7. In a healthy run the mismatch is highest at iteration 0 (the prior) and falls over the following steps, while `model_change` shrinks as the ensemble settles. Compare the two runs row by row, and look at how each metric moves from step to step, not only at the last one:

```bash
uv run apex summary work/linear_gaussian/runs/linear_esmda work/linear_gaussian/runs/linear_gies
```

`apex report` takes one run directory and prints its data mismatch history and performance metrics. The mismatch should fall from the prior (iteration 0) and then level off. A mismatch that keeps rising, or does not move at all, is a sign to revisit the configuration. ES-MDA takes a fixed number of steps, while GIES stops by its own criteria, so the histories differ in length.

```bash
uv run apex report work/linear_gaussian/runs/linear_esmda
uv run apex report work/linear_gaussian/runs/linear_gies
```

With `--plots` it also writes the result figures (PNG and PDF), and `--output` takes the folder to write them to (the default is a `report` folder inside the run directory). This writes the ES-MDA figures to a folder named `figs`, in the folder you run the command from:

```bash
uv run apex report work/linear_gaussian/runs/linear_esmda --plots --output figs
```

### Starting, stopping and continuing runs

This notebook started its runs from Python with `submit_run`. From the terminal the equivalent is `apex run CONFIG`, which takes a configuration file and starts a run. The run directory is `runs/<name>` next to the configuration file, with `<name>` taken from `[run]` `name`, the same place as the notebook's runs.

To keep a terminal run apart from the notebook's runs, the next cell writes a copy of the ES-MDA configuration with another `[run]` name, next to the original so that its data files are found.

In [11]:
short = copied.folder / "config_esmda_short.toml"
text = config.read_text()
if 'name = "linear_esmda"' not in text:
    raise ValueError("the run name line was not found in the configuration")
short.write_text(text.replace('name = "linear_esmda"', 'name = "linear_esmda_short"', 1))
print(short.relative_to(Path.cwd()))

work/linear_gaussian/config_esmda_short.toml


An *analysis step* is one update of the ensemble against the data. For ES-MDA it is the same thing as one iteration, so a configuration with four `inflation` entries has four analysis steps.

A *checkpoint* is the state of the run saved after an analysis step. A run that stops, on purpose or by interruption, can continue from its last checkpoint instead of starting over. The run directory's `checkpoints` folder holds them.

`--max-steps` on `apex run` is the reproducible way to stop: the run ends cleanly after that many analysis steps. Here we run two of the four steps, look at the state, then continue with `apex resume`, which takes the run directory. Without `--max-steps` the resumed run carries on until it ends. Then we look again:

```bash
uv run apex run work/linear_gaussian/config_esmda_short.toml --max-steps 2
uv run apex summary work/linear_gaussian/runs/linear_esmda_short
uv run apex resume work/linear_gaussian/runs/linear_esmda_short
uv run apex summary work/linear_gaussian/runs/linear_esmda_short
```

The first `apex summary` should list the prior (iteration 0) and two steps, and the second should list all four steps after the resume.

For a long run you did not limit in advance, `apex stop RUN_DIR` asks a run that is still going to stop at its next checkpoint, that is, when its current analysis step is done. Run it from a second terminal while the first one is busy with `apex run`. Then use `apex summary RUN_DIR` to see which iterations were saved, and `apex resume RUN_DIR` to continue from there. These example runs are so short that they end before you could catch them, which is why we used `--max-steps`.

## Pitfalls

- **Skipping validation.** Validate first. It catches configuration mistakes before any simulator time is spent.
- **Ignoring the plan.** The maximum number of simulator runs is the most you can pay, so read it before you start. For GIES it is a worst-case ceiling, and the run may use much less.
- **Reading a summary of a run that did not end in `complete`.** Print the state first.
- **Reading `rmse` for a real case.** It is scored against a known true model, which only a synthetic case has.
- **Choosing a method from one metric.** Read the mismatch together with the mismatch history and, for real cases, the forecasts. Raw and normalized mismatch can rank runs differently.
- **Trusting defaults instead of the file.** Check the values written in your configuration, such as the GIES damping options.
- **Searching for a directory named after the run id.** Run directories are named after the `[run]` name; take the path from the run summary (`path`).
- **Running the terminal commands from another folder.** The paths above start at `work`, so run them from the folder where the notebook ran.
- **Re-running a finished configuration.** Its run directory already exists. Resume it, or run a copy with another `[run]` name.

## What to read next

- Change `ensemble_size` or the `inflation` list in a copy of the configuration, validate, and run it under another `[run]` name to see how the plan and the metrics change.
- Open the figures that the `apex report --plots --output figs` command wrote to the `figs` folder.
- Look through the examples catalogue for problems closer to your own.
- Forecasting beyond history is outside this tutorial, and this example does not include a prediction configuration. `apex predict` takes a prediction configuration (a TOML file) for a finished history-matching run and forecasts its ensembles. Its options are listed by:

```bash
uv run apex predict --help
```